## 准备测试数据

In [1]:
from pymilvus import AsyncMilvusClient, DataType
client = AsyncMilvusClient(
    uri="http://127.0.0.1:19530",
    db_name="deal_pilot",
)
collection_name = "analyzer_hybrid_demo"

In [2]:

from deal_pilot.rag.embedding_model import create_embedding_model

documents = [
    {
        "id": 1,
        "text": "ERR4217：PKIX 路径构建错误；TLS_ALERT_CERTIFICATE_UNKNOWN；CA 中间证书轮换；请替换 intermediate.pem 并重启服务。"
    },
    {
        "id": 2,
        "text": "ERR4218：客户端请求没有得到响应。检查服务进程是否正常、入口防火墙是否允许流量，以及路由端口是否开放。"
    },
    {
        "id": 3,
        "text": "ERR5217：访问凭证已失效。重新签发 API token，再重新提交请求。"
    },
    {
        "id": 4,
        "text": "网关调用长时间没有返回：核对名称解析、出口路由和链路延迟。"
    },
    {
        "id": 5,
        "text": "远端实例无法到达：检查应用健康状态、网络 ACL 和负载均衡配置。"
    },
    {
        "id": 6,
        "text": "会话协商卡住时，查看 TLS 日志、证书链和中间代理转发规则。"
    },
    {
        "id": 7,
        "text": "TLS 握手失败，通常需要检查服务端证书是否过期、证书链是否完整。"
    },
    {
        "id": 8,
        "text": "PKIX 路径构建失败时，请验证根 CA 和中间 CA 是否都在信任列表中。"
    },
    {
        "id": 9,
        "text": "收到 unknown CA 告警后，确认客户端信任的证书颁发机构与服务端证书一致。"
    },
    {
        "id": 10,
        "text": "客户端连接超时可能由代理、DNS 解析或网络链路延迟造成。"
    },
]

embedding_model = create_embedding_model()
vectors = await embedding_model.aembed_documents([doc["text"] for doc in documents])
entities = [
    {**doc, "vector": vector}
    for doc, vector in zip(documents, vectors)
]



if collection_name not in await client.list_collections():
    schema = client.create_schema(auto_id=False)
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("text", DataType.VARCHAR, max_length=2048)
    schema.add_field("vector", DataType.FLOAT_VECTOR, dim=1024)

    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="vector", index_type="HNSW", metric_type="COSINE"
    )
    await client.create_collection(
        collection_name=collection_name, schema=schema, index_params=index_params
    )

await client.upsert(collection_name=collection_name, data=entities)
await client.flush(collection_name=collection_name)
print(f"已写入 {len(entities)} 条数据：deal_pilot.{collection_name}")

已写入 10 条数据：deal_pilot.analyzer_hybrid_demo


试着模拟运维人员查询错误码：ERR4217

有些时候，通过全文检索的精确匹配的结果反而优于向量化的语义检索

全文检索由两个关键步骤组成：

1. **分析器**: 将文本拆分为关键词
2. **索引与召回**: 将关键词存储到索引中，并在查询时进行召回

# 分析器

分析器可以将文本拆分为关键词

https://milvus.io/docs/zh/analyzer-overview.md

In [15]:
text = "ERR4217：PKIX 路径构建错误；TLS_ALERT_CERTIFICATE_UNKNOWN；CA 中间证书轮换；请替换 intermediate.pem 并重启服务。"

result = await client.run_analyzer(
    texts=text,
    analyzer_params={
        "type": "chinese",
        "stop_words": ["请", "并"]
    }
)
result

['ERR4217', 'PKIX', '路径', '构建', '错误', 'TLS', 'ALERT', 'CERTIFICATE', 'UNKNOWN', 'CA', '中间', '证书', '轮换', '替换', 'intermediate', 'pem', '重启', '服务']